# Postprocessing and Analysis

This notebook provides comprehensive postprocessing tools for analyzing:
- Super-resolution results
- Autoencoder latent spaces
- Flow physics (vorticity, strain rate, Q-criterion)
- Statistical comparisons

**Note**: This notebook is for visualization and analysis only. All data fetching and model training should be done using Python scripts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
from pathlib import Path
import json

# Set plotting defaults
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

print("Libraries loaded successfully")

## 1. Load Data

Load the cached plane data and/or ML results.

In [ ]:
# Configuration
DATA_DIR = Path("../data")
CACHE_DIR = DATA_DIR / "raw" / "channel_plane_256"

# List available cached planes
if CACHE_DIR.exists():
    planes = list(CACHE_DIR.glob("*.npz"))
    print(f"Found {len(planes)} cached planes:")
    for p in planes:
        print(f"  - {p.name}")
else:
    print(f"Cache directory not found: {CACHE_DIR}")
    print("Run data fetching first: python -m dataloaders.jhtdb_fetcher")

In [ ]:
# Load a specific plane
plane_path = planes[0] if planes else None

if plane_path:
    print(f"Loading: {plane_path.name}")
    
    with np.load(plane_path) as npz:
        data = npz['data']
        primary_coords = npz['primary_values']
        secondary_coords = npz['secondary_values']
    
    meta_path = plane_path.with_suffix('.json')
    with open(meta_path, 'r') as f:
        metadata = json.load(f)
    
    components = metadata['components']
    
    print(f"\nData shape: {data.shape}")
    print(f"Components: {components}")
    print(f"Primary axis range: [{primary_coords.min():.3f}, {primary_coords.max():.3f}]")
    print(f"Secondary axis range: [{secondary_coords.min():.3f}, {secondary_coords.max():.3f}]")

## 2. Compute Vorticity and Derived Quantities

In [ ]:
# Add parent directory to path to import utils
import sys
sys.path.append('..')

from utils.physics import compute_vorticity_2d, compute_q_criterion_2d, compute_enstrophy

In [ ]:
# Extract velocity components
ux = data[0]  # Assuming first component is ux
uy = data[1]  # Assuming second component is uy

# Compute grid spacing
dx = float(np.mean(np.diff(primary_coords)))
dy = float(np.mean(np.diff(secondary_coords)))

print(f"Grid spacing: dx={dx:.6f}, dy={dy:.6f}")

# Compute vorticity
omega_z = compute_vorticity_2d(ux, uy, dx, dy)

print(f"\nVorticity computed:")
print(f"  Shape: {omega_z.shape}")
print(f"  Min: {omega_z.min():.3f}")
print(f"  Max: {omega_z.max():.3f}")
print(f"  Mean: {omega_z.mean():.3f}")
print(f"  Std: {omega_z.std():.3f}")

# Compute Q-criterion
Q = compute_q_criterion_2d(ux, uy, dx, dy)

print(f"\nQ-criterion computed:")
print(f"  Positive Q regions: {(Q > 0).sum() / Q.size * 100:.1f}%")

# Compute enstrophy
enstrophy = compute_enstrophy(omega_z)
print(f"\nEnstrophy: {enstrophy:.6f}")

## 3. Visualization

Create publication-quality visualizations.

In [ ]:
def plot_field_comparison(fields, titles, cmaps=None, figsize=(15, 5), vmin=None, vmax=None):
    """
    Plot multiple fields side by side.
    
    Args:
        fields: List of 2D arrays
        titles: List of titles
        cmaps: List of colormaps (default: 'RdBu_r' for all)
        figsize: Figure size
        vmin: Min value for colormap (default: auto per field)
        vmax: Max value for colormap (default: auto per field)
    """
    n = len(fields)
    if cmaps is None:
        cmaps = ['RdBu_r'] * n
    
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    
    for ax, field, title, cmap in zip(axes, fields, titles, cmaps):
        if vmin is None or vmax is None:
            vmin_field = field.min()
            vmax_field = field.max()
        else:
            vmin_field = vmin
            vmax_field = vmax
        
        im = ax.imshow(field.T, origin='lower', cmap=cmap, 
                      vmin=vmin_field, vmax=vmax_field, aspect='auto')
        ax.set_title(title)
        ax.set_xlabel('x')
        ax.set_ylabel('z')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    return fig

In [ ]:
# Plot velocity components
fig = plot_field_comparison(
    [ux, uy],
    ['u_x', 'u_y'],
    cmaps=['viridis', 'viridis'],
    figsize=(12, 5)
)
plt.suptitle('Velocity Components', y=1.02, fontsize=14)
plt.show()

In [ ]:
# Plot vorticity
fig = plot_field_comparison(
    [omega_z],
    ['Vorticity $\omega_z$'],
    cmaps=['RdBu_r'],
    figsize=(8, 6)
)
plt.show()

In [ ]:
# Plot Q-criterion
fig = plot_field_comparison(
    [Q],
    ['Q-criterion'],
    cmaps=['seismic'],
    figsize=(8, 6)
)
plt.show()

## 4. Statistical Analysis

In [ ]:
def plot_histogram(data, title, bins=50, figsize=(10, 5)):
    """
    Plot histogram with statistics.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    
    # Histogram
    ax1.hist(data.ravel(), bins=bins, density=True, alpha=0.7, edgecolor='black')
    ax1.axvline(data.mean(), color='r', linestyle='--', linewidth=2, label=f'Mean: {data.mean():.3f}')
    ax1.set_xlabel('Value')
    ax1.set_ylabel('Probability Density')
    ax1.set_title(f'{title} - Histogram')
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    # Q-Q plot
    from scipy import stats
    stats.probplot(data.ravel(), dist="norm", plot=ax2)
    ax2.set_title(f'{title} - Q-Q Plot')
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    return fig

In [ ]:
# Plot vorticity statistics
fig = plot_histogram(omega_z, 'Vorticity', bins=100)
plt.show()

## 5. Super-Resolution Results Analysis

Load and compare SR predictions with ground truth.

In [ ]:
# Load SR dataset
h5_path = DATA_DIR / "channel_sr_256.h5"

if h5_path.exists():
    with h5py.File(h5_path, 'r') as f:
        # Load a few validation samples
        n_samples = min(5, f['val/x'].shape[0])
        
        lr_samples = f['val/x'][:n_samples]
        hr_samples = f['val/y'][:n_samples]
        
        # Load metadata
        print("Dataset attributes:")
        for key in f.attrs.keys():
            print(f"  {key}: {f.attrs[key]}")
        
        print(f"\nLoaded {n_samples} validation samples")
        print(f"LR shape: {lr_samples.shape}")
        print(f"HR shape: {hr_samples.shape}")
else:
    print(f"Dataset not found: {h5_path}")
    print("Run: python -m dataloaders.dataset_builder --config configs/channel_plane_256.yaml")

In [ ]:
# Visualize LR/HR pairs
if h5_path.exists():
    idx = 0  # Sample to visualize
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # LR
    im0 = axes[0].imshow(lr_samples[idx, 0], cmap='viridis')
    axes[0].set_title('Low Resolution (Input)')
    plt.colorbar(im0, ax=axes[0])
    
    # HR
    im1 = axes[1].imshow(hr_samples[idx, 0], cmap='viridis')
    axes[1].set_title('High Resolution (Target)')
    plt.colorbar(im1, ax=axes[1])
    
    plt.tight_layout()
    plt.show()

## 6. Spectral Analysis

Analyze energy spectra and turbulence scales.

In [ ]:
def compute_2d_spectrum(field):
    """
    Compute 2D power spectrum.
    """
    # FFT
    fft_field = np.fft.fft2(field)
    power = np.abs(fft_field) ** 2
    
    # Get wavenumbers
    kx = np.fft.fftfreq(field.shape[0], d=dx)
    ky = np.fft.fftfreq(field.shape[1], d=dy)
    
    # Compute radial spectrum
    kx_grid, ky_grid = np.meshgrid(kx, ky, indexing='ij')
    k_mag = np.sqrt(kx_grid**2 + ky_grid**2)
    
    return k_mag, power

def radial_spectrum(k_mag, power, n_bins=50):
    """
    Compute radially averaged spectrum.
    """
    k_flat = k_mag.ravel()
    power_flat = power.ravel()
    
    k_bins = np.linspace(0, k_flat.max(), n_bins)
    k_centers = 0.5 * (k_bins[1:] + k_bins[:-1])
    
    spectrum = np.zeros(len(k_centers))
    
    for i in range(len(k_centers)):
        mask = (k_flat >= k_bins[i]) & (k_flat < k_bins[i+1])
        if mask.sum() > 0:
            spectrum[i] = power_flat[mask].mean()
    
    return k_centers, spectrum

In [ ]:
# Compute spectrum for vorticity
k_mag, power = compute_2d_spectrum(omega_z)
k_radial, spectrum = radial_spectrum(k_mag, power, n_bins=50)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

# Remove zero wavenumber
mask = k_radial > 0
ax.loglog(k_radial[mask], spectrum[mask], 'o-', linewidth=2, markersize=4)

# Reference slopes
k_ref = k_radial[mask][len(k_radial[mask])//3:]
E_ref_53 = spectrum[mask][len(spectrum[mask])//3] * (k_ref / k_ref[0])**(-5/3)
ax.loglog(k_ref, E_ref_53, 'k--', alpha=0.5, linewidth=2, label='$k^{-5/3}$ (Kolmogorov)')

ax.set_xlabel('Wavenumber $k$', fontsize=12)
ax.set_ylabel('Energy $E(k)$', fontsize=12)
ax.set_title('Energy Spectrum', fontsize=14)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

## 7. Export Results

Save processed data and figures.

In [ ]:
# Create output directory
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Save vorticity field
np.save(OUTPUT_DIR / "vorticity.npy", omega_z)
print(f"Saved vorticity to {OUTPUT_DIR / 'vorticity.npy'}")

# Save Q-criterion
np.save(OUTPUT_DIR / "q_criterion.npy", Q)
print(f"Saved Q-criterion to {OUTPUT_DIR / 'q_criterion.npy'}")

# Save statistics
stats = {
    'enstrophy': float(enstrophy),
    'omega_mean': float(omega_z.mean()),
    'omega_std': float(omega_z.std()),
    'omega_min': float(omega_z.min()),
    'omega_max': float(omega_z.max()),
}

with open(OUTPUT_DIR / "statistics.json", 'w') as f:
    json.dump(stats, f, indent=2)

print(f"\nSaved statistics to {OUTPUT_DIR / 'statistics.json'}")
print(json.dumps(stats, indent=2))

## Summary

This notebook demonstrated:
1. Loading cached JHTDB data
2. Computing derived flow quantities (vorticity, Q-criterion)
3. Comprehensive visualization
4. Statistical analysis
5. Spectral analysis
6. Exporting results

For training models, use the command-line scripts:
```bash
# Fetch data
python -m dataloaders.jhtdb_fetcher --config configs/channel_plane_256.yaml

# Build dataset
python -m dataloaders.dataset_builder --config configs/channel_plane_256.yaml

# Train SR model
python -m models.trainers --config configs/channel_plane_256.yaml --task sr
```